# ПЗ 7 — Распознавание объектов через LLM (Qwen / Gemini API)

**Задача:** отправить кадры в multimodal LLM (Qwen через OpenAI-compatible API или Google Gemini), получить описание сцены в структурированном виде. Развернуть FastAPI-сервис на Timeweb VM.

**Стек:** `openai` (для Qwen), `google-generativeai` (для Gemini), `fastapi`, `uvicorn`

In [ ]:
!pip install openai google-generativeai fastapi uvicorn python-multipart -q

## Вариант A: Qwen через OpenAI-compatible API

In [ ]:
import os, base64, json
from openai import OpenAI

# Вставьте свой API ключ Qwen / DashScope
QWEN_API_KEY = 'sk-ВАШИ_КЛЮЧ_QWEN'

client = OpenAI(
    api_key=QWEN_API_KEY,
    base_url='https://dashscope.aliyuncs.com/compatible-mode/v1',
)

def encode_image(path: str) -> str:
    with open(path, 'rb') as f:
        return base64.b64encode(f.read()).decode('utf-8')

def describe_frame_qwen(image_path: str) -> dict:
    b64 = encode_image(image_path)
    resp = client.chat.completions.create(
        model='qwen-vl-max',
        messages=[{
            'role': 'user',
            'content': [
                {'type': 'image_url', 'image_url': {'url': f'data:image/jpeg;base64,{b64}'}},
                {'type': 'text', 'text': (
                    'Опиши объекты на кадре. '
                    'Верни JSON: {"objects": [...], "scene": "...", "mood": "..."}'
                )},
            ],
        }],
        max_tokens=512,
    )
    try:
        return json.loads(resp.choices[0].message.content)
    except Exception:
        return {'raw': resp.choices[0].message.content}

## Вариант B: Google Gemini API

In [ ]:
import google.generativeai as genai
from PIL import Image

# Вставьте свой Google API Key
GEMINI_API_KEY = 'ВАШИ_КЛЮЧ_GEMINI'
genai.configure(api_key=GEMINI_API_KEY)
gemini = genai.GenerativeModel('gemini-1.5-flash')

def describe_frame_gemini(image_path: str) -> dict:
    img = Image.open(image_path)
    prompt = (
        'Опиши объекты на кадре. '
        'Верни JSON: {"objects": [...], "scene": "...", "mood": "..."}'
    )
    resp = gemini.generate_content([img, prompt])
    try:
        text = resp.text.strip().lstrip('```json').rstrip('```')
        return json.loads(text)
    except Exception:
        return {'raw': resp.text}

In [ ]:
# Прогнать несколько кадров
FRAMES_DIR = '/content/outputs/frames'
OUTPUT_DIR = '/content/outputs/detections'
os.makedirs(OUTPUT_DIR, exist_ok=True)

frames = sorted(f for f in os.listdir(FRAMES_DIR) if f.endswith('.jpg'))[:10]
llm_results = []

for fname in frames:
    path = os.path.join(FRAMES_DIR, fname)
    # Используйте describe_frame_qwen или describe_frame_gemini
    result = describe_frame_gemini(path)
    result['frame'] = fname
    llm_results.append(result)
    print(f'{fname}: {result}')

with open(f'{OUTPUT_DIR}/llm_detections.json', 'w', encoding='utf-8') as f:
    json.dump(llm_results, f, ensure_ascii=False, indent=2)
print('Сохранено в llm_detections.json')

## FastAPI-сервис (запуск на Timeweb VM)

См. файл `docker/Dockerfile` и `docker/docker-compose.yml` в репозитории.

Быстрый запуск локально:
```bash
cd docker
docker-compose up -d
# Сервис доступен на http://localhost:8000
# POST /analyze — отправить изображение, получить JSON с описанием
```